# 🚀 Neural Network Deployment — From PyTorch to Production
## The Complete Autonomous Vehicle/Robotics Deployment Pipeline

**A companion notebook to [Neural Optimization](https://thinkautonomous.ai) by [Think Autonomous](https://thinkautonomous.ai)**

This notebook covers the **real deployment pipeline** used in production autonomous systems like [Autoware](https://github.com/autowarefoundation/autoware):
1. Load a robotics perception model → visualize → baseline benchmark
2. Export to ONNX → validate → compare FPS with ONNX Runtime
3. Understand the PyTorch ↔ ONNX workflow
4. Optimize before export: Pruning + Quantization → ONNX
5. Analyze Autoware's **TensorRT C++ production code**
6. Deploy with TensorRT on Colab (FP16 inference)
7. Production profiling & benchmarking
8. End-to-end pipeline project: Train → Optimize → Export → Deploy → Benchmark

**Requirements:** Google Colab with a **T4 GPU** runtime (recommended)

---
## 0. Environment Setup

In [ ]:
# Core ML packages — modern versions all support numpy 2.x
!pip install torch torchvision onnx onnxsim onnxruntime-gpu Pillow matplotlib -q
!pip install openvino -q
!pip install tensorrt pycuda -q

import warnings
warnings.filterwarnings('ignore')
print("✓ Environment ready")

In [ ]:
import torch
import torchvision
import torchvision.transforms as T
import torch.nn as torch_nn
import numpy as np
import time
import os
from PIL import Image
import matplotlib.pyplot as plt
import urllib.request
import copy

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:             {torch.cuda.get_device_name(0)}')
    print(f'CUDA version:    {torch.version.cuda}')
else:
    print('⚠️  No GPU detected. TensorRT section will be CPU-only.')
    print('   For full GPU support, use Runtime → Change runtime type → T4 GPU')


---
## 1. Load a Robotics Perception Model, Visualize & Benchmark

### Context: Autonomous Driving Perception
Perception is the first step in autonomous systems. Deep learning models predict:
- **Semantic segmentation** → "What is each pixel?"
- **Object detection** → "Where are the cars/pedestrians?"
- **Depth estimation** → "How far is that object?"

We'll use **DeepLabV3-MobileNetV3-Large**: a real segmentation model used in mobile/edge AV systems.

**Real Autoware example:** [Autoware's EgoLanes model](https://github.com/autowarefoundation/autoware/tree/main/perception/perception_lanelet2_map_based) predicts lane lines for autonomous driving.

In [ ]:
# Load pretrained DeepLabV3 with MobileNetV3 backbone
# Used in real autonomous driving systems for road/obstacle/sky segmentation
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

model = torchvision.models.segmentation.deeplabv3_mobilenet_v3_large(pretrained=True)
model.eval().to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model: DeepLabV3-MobileNetV3-Large')
print(f'Total parameters: {total_params:,}')
print(f'Model size: ~{total_params * 4 / 1024 / 1024:.1f} MB (FP32)')
print(f'Backbone: MobileNetV3-Large (lightweight, suitable for mobile/edge)')

In [ ]:
# Download a real urban driving scene (bus + pedestrians — perfect for AV segmentation)
# This is the standard YOLO / ultralytics test image used across AV benchmarks
IMG_URL  = 'https://ultralytics.com/images/bus.jpg'
IMG_PATH = 'driving_scene.jpg'

if not os.path.exists(IMG_PATH):
    urllib.request.urlretrieve(IMG_URL, IMG_PATH)

original_image = Image.open(IMG_PATH).convert('RGB')
print(f'Loaded: {IMG_PATH}  size={original_image.size}')

plt.figure(figsize=(12, 6))
plt.imshow(original_image)
plt.title('Input: Urban Driving Scene (bus, pedestrians, road)')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Preprocessing pipeline (same as production)
preprocess = T.Compose([
    T.Resize(520),
    T.CenterCrop(480),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

input_tensor = preprocess(original_image).unsqueeze(0).to(device)
print(f'Input tensor shape: {input_tensor.shape}  device={input_tensor.device}')

# Cityscapes color palette
CITYSCAPES_COLORS = np.array([
    [0, 0, 0],       # background
    [128, 64, 128],  # road
    [244, 35, 232],  # sidewalk
    [70, 70, 70],    # building
    [102, 102, 156], # wall
    [190, 153, 153], # fence
    [153, 153, 153], # pole
    [250, 170, 30],  # traffic light
    [220, 220, 0],   # traffic sign
    [107, 142, 35],  # vegetation
    [152, 251, 152], # terrain
    [70, 130, 180],  # sky
    [220, 20, 60],   # person
    [255, 0, 0],     # rider
    [0, 0, 142],     # car
    [0, 0, 70],      # truck
    [0, 60, 100],    # bus
    [0, 80, 100],    # train
    [0, 0, 230],     # motorcycle
    [119, 11, 32],   # bicycle
    [128, 128, 128], # other
], dtype=np.uint8)

def visualize_segmentation(prediction, title='Segmentation'):
    """Convert model output to color segmentation map."""
    if isinstance(prediction, torch.Tensor):
        seg_map = prediction.argmax(dim=1).squeeze().cpu().numpy()
    else:
        seg_map = prediction.argmax(axis=1).squeeze()

    h, w = seg_map.shape
    color_map = np.zeros((h, w, 3), dtype=np.uint8)
    for cls_id in range(len(CITYSCAPES_COLORS)):
        color_map[seg_map == cls_id] = CITYSCAPES_COLORS[cls_id]

    return color_map, seg_map

print('✓ Preprocessing ready')


In [ ]:
# Run PyTorch inference (baseline)
with torch.no_grad():
    pytorch_output = model(input_tensor)['out']

# Move to CPU for visualization / numpy ops
pytorch_output_cpu = pytorch_output.cpu()
pytorch_seg, pytorch_classes = visualize_segmentation(pytorch_output_cpu)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(T.CenterCrop(480)(T.Resize(520)(original_image)))
axes[0].set_title('Input Image', fontsize=14)
axes[0].axis('off')
axes[1].imshow(pytorch_seg)
axes[1].set_title('Semantic Segmentation Output', fontsize=14)
axes[1].axis('off')
plt.tight_layout()
plt.show()

unique_classes = np.unique(pytorch_classes)
print(f'Classes detected: {len(unique_classes)}')
print(f'Output shape: {pytorch_output.shape}')
print(f'Device: {pytorch_output.device}')


In [ ]:
# Benchmark function — GPU-aware (syncs CUDA before stopping the clock)
_use_cuda = torch.cuda.is_available()

def benchmark(run_fn, name, n_warmup=10, n_runs=50):
    for _ in range(n_warmup):
        run_fn()
    if _use_cuda:
        torch.cuda.synchronize()

    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        run_fn()
        if _use_cuda:
            torch.cuda.synchronize()   # wait for GPU kernel to finish
        times.append((time.perf_counter() - start) * 1000)

    avg, std = np.mean(times), np.std(times)
    fps = 1000 / avg
    print(f'{name:.<45} {avg:.1f} ms ± {std:.1f} ms  ({fps:.1f} FPS)')
    return avg

hw = f'GPU ({torch.cuda.get_device_name(0)})' if _use_cuda else 'CPU'
pytorch_time = benchmark(
    lambda: model(input_tensor),
    f'PyTorch ({hw})'
)

---
## 2. Export to ONNX → Validate → Load with ONNX Runtime → Compare FPS

### Why ONNX?
- **Framework-agnostic** → Train in PyTorch, deploy anywhere
- **Optimized runtimes** → ONNX Runtime, TensorRT, OpenVINO all optimize ONNX
- **Model zoo** → Download pre-converted models

**Autoware example:** Most Autoware perception models are exported to ONNX for deployment across different hardware.

In [ ]:
import onnx
import onnxsim

ONNX_PATH = 'deeplabv3_mobilenetv3.onnx'

# DeepLabV3 returns a dict, wrap it to export cleanly
class SegmentationWrapper(torch_nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        return self.model(x)['out']

wrapped_model = SegmentationWrapper(model)
wrapped_model.eval()

print('Exporting to ONNX (opset 18, for ONNX Runtime)...')
torch.onnx.export(
    wrapped_model,
    input_tensor,
    ONNX_PATH,
    verbose=False,
    input_names=['input'],
    output_names=['output'],
    opset_version=18,
)

# Validate
model_onnx = onnx.load(ONNX_PATH)
onnx.checker.check_model(model_onnx)
print('✓ ONNX model validated')

# Simplify
print('Simplifying ONNX graph...')
model_simplified, ok = onnxsim.simplify(model_onnx)
if ok:
    onnx.save(model_simplified, ONNX_PATH)
    print('✓ ONNX model simplified')

size_mb = os.path.getsize(ONNX_PATH) / 1024 / 1024
print(f'\nONNX model saved: {ONNX_PATH} ({size_mb:.1f} MB)')

# ── TRT-specific export ──────────────────────────────────────────────────────
# TensorRT 8.x only supports ONNX opset ≤17 and chokes on ops emitted by
# PyTorch's new dynamo exporter. We export a second file with the legacy
# TorchScript exporter (dynamo=False) at opset 17 specifically for TRT.
ONNX_TRT_PATH = 'deeplabv3_for_trt.onnx'
print('\nExporting TRT-compatible ONNX (opset 17, legacy exporter)...')
try:
    torch.onnx.export(
        wrapped_model, input_tensor, ONNX_TRT_PATH,
        input_names=['input'], output_names=['output'],
        opset_version=17, dynamo=False,
    )
except TypeError:
    # PyTorch < 2.1 — legacy exporter is already the default
    torch.onnx.export(
        wrapped_model, input_tensor, ONNX_TRT_PATH,
        input_names=['input'], output_names=['output'],
        opset_version=17,
    )
trt_onnx = onnx.load(ONNX_TRT_PATH)
onnx.checker.check_model(trt_onnx)
trt_onnx_simplified, ok = onnxsim.simplify(trt_onnx)
if ok:
    onnx.save(trt_onnx_simplified, ONNX_TRT_PATH)
print(f'✓ TRT ONNX saved: {ONNX_TRT_PATH}')


In [ ]:
import onnxruntime as ort

print(f'ONNX Runtime version: {ort.__version__}')
print(f'Available providers: {ort.get_available_providers()}')

# Create inference session - selects best provider automatically
providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
session = ort.InferenceSession(ONNX_PATH, providers=providers)

active_provider = session.get_providers()[0]
print(f'\nActive provider: {active_provider}')

# Get input/output metadata
input_meta = session.get_inputs()[0]
output_meta = session.get_outputs()[0]
print(f'Input:  {input_meta.name} {input_meta.shape}')
print(f'Output: {output_meta.name} {output_meta.shape}')


In [ ]:
# Run ONNX Runtime inference
# ORT always takes numpy on CPU — move off GPU first
onnx_input = input_tensor.cpu().numpy()
onnx_output = session.run([output_meta.name], {input_meta.name: onnx_input})[0]

onnx_seg, _ = visualize_segmentation(onnx_output)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(pytorch_seg)
axes[0].set_title('PyTorch Output', fontsize=14)
axes[0].axis('off')
axes[1].imshow(onnx_seg)
axes[1].set_title(f'ONNX Runtime Output ({session.get_providers()[0]})', fontsize=14)
axes[1].axis('off')
plt.suptitle('Output Comparison: PyTorch vs ONNX Runtime', fontsize=14)
plt.tight_layout()
plt.show()

pytorch_np = pytorch_output_cpu.numpy()
max_diff = np.max(np.abs(pytorch_np - onnx_output))
print(f'Max numerical difference: {max_diff:.6f}')
print(f'Outputs match: ✓' if max_diff < 0.001 else 'Outputs differ ⚠')
print(f'ORT provider: {session.get_providers()[0]}')


In [ ]:
# Benchmark ONNX Runtime
onnx_time = benchmark(
    lambda: session.run([output_meta.name], {input_meta.name: onnx_input}),
    f'ONNX Runtime ({active_provider})'
)


---
## 3. The PyTorch ↔ ONNX Workflow

### The Training/Deployment Split
```
Development (Research)           Production (Deployment)
├─ PyTorch (flexible)            ├─ ONNX (optimized)
├─ Easy debugging                ├─ Framework-agnostic
├─ Rich ecosystem                ├─ Multiple runtimes
└─ Custom training loops         └─ Fast inference

Workflow:
  Train in PyTorch → Export to ONNX → Deploy with ORT/TensorRT/OpenVINO
```

### Real Autoware Example
In Autoware's perception pipeline:
- **Development:** Models trained in PyTorch (like EgoLanes, AutoSteer)
- **Export:** `torch.onnx.export()` to create ONNX files
- **Deployment:** C++ nodes load ONNX, run inference with TensorRT/ORT

Let's walk through a complete example with a **steering prediction model** (similar to Autoware's AutoSteer).

In [ ]:
# Define a simple steering prediction network (inspired by Autoware's AutoSteer)
class SteeringPredictor(torch_nn.Module):
    """Predicts steering angle from an image."""
    def __init__(self):
        super().__init__()
        # Backbone: simple CNN
        self.features = torch_nn.Sequential(
            torch_nn.Conv2d(3, 32, kernel_size=5, stride=2, padding=2),
            torch_nn.ReLU(inplace=True),
            torch_nn.MaxPool2d(2, 2),
            torch_nn.Conv2d(32, 64, kernel_size=5, stride=2, padding=2),
            torch_nn.ReLU(inplace=True),
            torch_nn.MaxPool2d(2, 2),
            torch_nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            torch_nn.ReLU(inplace=True),
        )
        # Prediction head: steering angle classification (61 classes: -30 to +30 degrees)
        self.classifier = torch_nn.Sequential(
            torch_nn.Linear(128 * 15 * 15, 256),
            torch_nn.ReLU(inplace=True),
            torch_nn.Dropout(0.5),
            torch_nn.Linear(256, 61),  # 61 steering angle bins
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

# Create and "train" the model
steering_model = SteeringPredictor()
steering_model.eval()

print(f'Steering model parameters: {sum(p.numel() for p in steering_model.parameters()):,}')
print(f'Output: 61 steering angle classes (-30° to +30°)')


In [ ]:
# Quick "training" loop (just to simulate model development)
steering_model.train()
optimizer = torch.optim.Adam(steering_model.parameters(), lr=0.001)
criterion = torch_nn.CrossEntropyLoss()

print('Simulating training (2 epochs on synthetic data)...')
for epoch in range(2):
    # Synthetic batch: random images + random steering angles
    synthetic_images = torch.randn(8, 3, 480, 480)
    synthetic_angles = torch.randint(0, 61, (8,))

    optimizer.zero_grad()
    outputs = steering_model(synthetic_images)
    loss = criterion(outputs, synthetic_angles)
    loss.backward()
    optimizer.step()

    print(f'  Epoch {epoch+1}/2, Loss: {loss.item():.4f}')

steering_model.eval()
print('✓ Training complete')


In [ ]:
# Export steering model to ONNX
STEERING_ONNX_PATH = 'steering_predictor.onnx'

dummy_input = torch.randn(1, 3, 480, 480)

torch.onnx.export(
    steering_model,
    dummy_input,
    STEERING_ONNX_PATH,
    input_names=['image'],
    output_names=['steering_logits'],
    opset_version=18,
)

# Validate
onnx_steer = onnx.load(STEERING_ONNX_PATH)
onnx.checker.check_model(onnx_steer)
print('✓ Steering ONNX exported and validated')

size_mb = os.path.getsize(STEERING_ONNX_PATH) / 1024 / 1024
print(f'  File: {STEERING_ONNX_PATH} ({size_mb:.3f} MB)')


In [ ]:
# Load and run with ONNX Runtime
steer_session = ort.InferenceSession(STEERING_ONNX_PATH, providers=['CPUExecutionProvider'])
steer_input = steer_session.get_inputs()[0]
steer_output = steer_session.get_outputs()[0]

# Inference
def softmax(x):
    e = np.exp(x - np.max(x))  # numerically stable
    return e / e.sum()

test_image = torch.randn(1, 3, 480, 480).numpy()
steering_logits = steer_session.run([steer_output.name], {steer_input.name: test_image})[0]

predicted_angle = np.argmax(steering_logits[0]) - 30  # Convert class index to angle
probs = softmax(steering_logits[0])
print(f'\nONNX Runtime Steering Prediction:')
print(f'  Predicted angle: {predicted_angle}°')
print(f'  Confidence: {probs[np.argmax(probs)]:.2%}')
print(f'\n✓ PyTorch ↔ ONNX workflow complete!')


---
## 4. Optimize Before Export: Pruning + Quantization → ONNX

### Why optimize before export?
- Smaller model size
- Faster inference
- Same workflow (PyTorch → ONNX → Deploy)
- Can stack optimizations (Pruning + Quantization)

### Techniques
1. **Structured pruning** - Remove entire filters/channels
2. **Quantization** - Reduce precision (FP32 → INT8)
3. **Knowledge distillation** - Covered in Neural Optimization course

We'll optimize the **DeepLabV3 model** from Section 1.

In [ ]:
import torch.nn.utils.prune as prune

# Create a copy for pruning
pruned_model = copy.deepcopy(wrapped_model)
pruned_model.eval()

# Global unstructured L1 magnitude pruning:
# Zeros the 40% smallest weights across ALL Conv2d layers.
# Unlike structured pruning (which removes whole filters and collapses
# feature maps), unstructured pruning preserves tensor shapes so
# inference still produces meaningful output — just with added sparsity.
conv_params = [
    (m, 'weight')
    for m in pruned_model.modules()
    if isinstance(m, torch_nn.Conv2d)
]
prune.global_unstructured(conv_params, pruning_method=prune.L1Unstructured, amount=0.4)
for m, name in conv_params:   # make permanent
    prune.remove(m, name)

# Count non-zero parameters
total = sum(p.numel() for p in model.parameters())
original_nonzero = sum((p != 0).sum().item() for p in model.parameters())
pruned_nonzero   = sum((p != 0).sum().item() for p in pruned_model.parameters())

print(f'Unstructured L1 pruning: 40% of smallest weights zeroed')
print(f'Original non-zero params: {100*original_nonzero/total:.1f}%')
print(f'Pruned non-zero params:   {100*pruned_nonzero/total:.1f}%')
print(f'Sparsity achieved:        {100*(1 - pruned_nonzero/total):.1f}%')


In [ ]:
# Run inference with pruned model
with torch.no_grad():
    pruned_output = pruned_model(input_tensor).cpu()

pruned_seg, _ = visualize_segmentation(pruned_output)

# Compare
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(pytorch_seg)
axes[0].set_title('Original Model', fontsize=14)
axes[0].axis('off')
axes[1].imshow(pruned_seg)
axes[1].set_title('Pruned Model (40% unstructured L1)', fontsize=14)
axes[1].axis('off')
plt.suptitle('Pruning: Original vs Pruned Output', fontsize=14)
plt.tight_layout()
plt.show()

# Measure accuracy difference
match = (pytorch_output_cpu.argmax(dim=1) == pruned_output.argmax(dim=1)).float().mean().item()
print(f'Pixel agreement with original: {match*100:.1f}%')


In [ ]:
# Export pruned model to ONNX
# Note: unstructured pruning zeros weights but does NOT reduce ONNX file size —
# ONNX stores all tensors densely. For real size reduction, use quantization.
PRUNED_ONNX_PATH = 'deeplabv3_pruned.onnx'

torch.onnx.export(
    pruned_model,
    input_tensor,
    PRUNED_ONNX_PATH,
    input_names=['input'],
    output_names=['output'],
    opset_version=18,
)

model_pruned_onnx = onnx.load(PRUNED_ONNX_PATH)
model_pruned_simplified, _ = onnxsim.simplify(model_pruned_onnx)
onnx.save(model_pruned_simplified, PRUNED_ONNX_PATH)

original_size = os.path.getsize(ONNX_PATH) / 1024 / 1024
pruned_size   = os.path.getsize(PRUNED_ONNX_PATH) / 1024 / 1024
print(f'Original ONNX FP32: {original_size:.1f} MB')
print(f'Pruned  ONNX FP32:  {pruned_size:.1f} MB  ← same (zeros still stored densely)')


In [ ]:
# INT8 dynamic quantization via ONNX Runtime — real ~4x file size reduction
from onnxruntime.quantization import quantize_dynamic, QuantType

QUANT_ONNX_PATH = 'deeplabv3_int8.onnx'
quantize_dynamic(ONNX_PATH, QUANT_ONNX_PATH, weight_type=QuantType.QInt8)

quant_size = os.path.getsize(QUANT_ONNX_PATH) / 1024 / 1024
print(f'Original FP32: {original_size:.1f} MB')
print(f'INT8 quant:    {quant_size:.1f} MB  ({(1 - quant_size/original_size)*100:.0f}% smaller)')


In [ ]:
# Benchmark all three ONNX variants
pruned_session = ort.InferenceSession(PRUNED_ONNX_PATH, providers=['CPUExecutionProvider'])
pruned_input_meta  = pruned_session.get_inputs()[0]
pruned_output_meta = pruned_session.get_outputs()[0]

quant_session = ort.InferenceSession(QUANT_ONNX_PATH, providers=['CPUExecutionProvider'])
quant_input_meta  = quant_session.get_inputs()[0]
quant_output_meta = quant_session.get_outputs()[0]

pruned_onnx_time = benchmark(
    lambda: pruned_session.run([pruned_output_meta.name], {pruned_input_meta.name: onnx_input}),
    'ONNX Runtime (pruned FP32)'
)
quant_onnx_time = benchmark(
    lambda: quant_session.run([quant_output_meta.name],  {quant_input_meta.name: onnx_input}),
    'ONNX Runtime (INT8 quantized)'
)
print(f'\nINT8 speedup vs original ONNX: {onnx_time/quant_onnx_time:.2f}x')


---
## 5. Analyze Autoware's TensorRT C++ Production Code

### Overview: How Autoware Deploys Neural Networks

[Autoware](https://github.com/autowarefoundation/autoware) is the world's leading open-source autonomous driving stack. Its perception nodes use TensorRT for GPU inference.

**Autoware's Deployment Stack:**
```
Python (Training)     →  PyTorch Model
    ↓
Export              →  ONNX File
    ↓
C++ Perception Node →  TensorRT Engine
    ↓
ROS2 Message Bus    →  Steering/Speed commands
```

### The TrtCommon Class Pattern

Autoware uses a `TrtCommon` utility class to manage TensorRT engines. Here's the real C++ interface:

In [ ]:
# Display actual Autoware TrtCommon header (simplified)
autoware_trt_header = '''\n// From: autoware/perception/tensorrt_common/include/tensorrt_common/tensorrt_common.hpp

namespace tensorrt_common {

class TrtCommon {
public:
    // Constructor: Load or build a TensorRT engine from ONNX
    TrtCommon(
        const std::string& model_path,        // Path to .onnx or .trtengine
        const std::string& precision,         // "fp32", "fp16", "int8"
        std::unique_ptr<nvinfer1::IInt8Calibrator> calibrator = nullptr,
        const BatchConfig& batch_config = {1, 1, 1},
        const size_t max_workspace_size = (1 << 30)  // 1GB
    );

    // Methods
    bool loadEngine(const std::string& path);
    bool buildEngineFromOnnx(
        const std::string& onnx_path,
        const std::string& engine_path
    );
    
    void setInput(const int index, const nvinfer1::Dims& dims);
    bool enqueueV2(
        void** bindings,           // GPU pointers: [input, output, ...]
        cudaStream_t stream,       // CUDA stream for async execution
        cudaEvent_t* inputConsumed
    );
    
    // Get engine info
    nvinfer1::ICudaEngine* getEngine() { return engine_.get(); }
    int getMaxBatchSize() const { return max_batch_size_; }
};

}  // namespace tensorrt_common
'''

print(autoware_trt_header)
print('\n[This is real C++ code from Autoware's tensorrt_common package]')


In [ ]:
# Display actual Autoware perception node pattern
autoware_node_pattern = '''\n// From: autoware/perception/traffic_light_classifier/lib/classifier.cpp
// Simplified example of how a perception node uses TrtCommon

#include <tensorrt_common/tensorrt_common.hpp>
#include <rclcpp/rclcpp.hpp>

class TrafficLightClassifier : public rclcpp::Node {
private:
    std::unique_ptr<tensorrt_common::TrtCommon> trt_;
    cudaStream_t stream_;
    
    void* d_input_;   // GPU memory for input
    void* d_output_;  // GPU memory for output

public:
    TrafficLightClassifier() : rclcpp::Node("traffic_light_classifier") {
        // Load ONNX model, build TensorRT engine
        trt_ = std::make_unique<tensorrt_common::TrtCommon>(
            "/path/to/traffic_light.onnx",
            "fp16"  // Use FP16 for faster inference on T4/V100
        );
        
        cudaStreamCreate(&stream_);
        cudaMalloc(&d_input_, input_size_bytes);
        cudaMalloc(&d_output_, output_size_bytes);
    }
    
    void inferenceCallback(const sensor_msgs::msg::Image& image_msg) {
        // Preprocess image on GPU
        preprocessImage(image_msg, d_input_, stream_);
        
        // Run inference
        void* bindings[] = {d_input_, d_output_};
        trt_->enqueueV2(bindings, stream_, nullptr);
        
        // Copy result back to CPU
        std::vector<float> h_output(num_classes_);
        cudaMemcpyAsync(h_output.data(), d_output_, output_size_bytes,
                        cudaMemcpyDeviceToHost, stream_);
        cudaStreamSynchronize(stream_);
        
        // Publish result
        publishTrafficLightState(h_output);
    }
};
'''

print(autoware_node_pattern)
print('\n[This pattern is used in Autoware perception nodes]')
print('\nKey insights:')
print('1. Load ONNX → Build TRT engine (TrtCommon handles this)')
print('2. Allocate GPU memory for input/output')
print('3. Run inference async with CUDA streams')
print('4. Sync stream, copy result back to CPU')
print('5. Publish ROS2 message')


---
## 6. TensorRT on Colab: FP16 Inference

### TensorRT: Nvidia's High-Performance Inference Engine
- **Reads ONNX directly** (no conversion step)
- **Graph optimization** (layer fusion, memory optimization)
- **Precision modes** (FP32, FP16, INT8)
- **Fastest inference** on Nvidia GPUs

TensorRT is what Autoware and production AVs use for GPU inference.

In [ ]:
trt_available = False
try:
    import tensorrt as trt
    import pycuda.driver as cuda
    import pycuda.autoinit
    
    trt_available = True
    print(f'TensorRT version: {trt.__version__}')
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print('✓ TensorRT is available')
except ImportError:
    print('⚠ TensorRT not available')
    print('  To use TensorRT:')
    print('  1. Use Google Colab with GPU runtime')
    print('  2. pip install tensorrt pycuda')
    print('  3. Or use an Nvidia Docker container')


In [ ]:
if trt_available:
    TRT_LOGGER = trt.Logger(trt.Logger.WARNING)
    TRT_MAJOR = int(trt.__version__.split('.')[0])

    def build_trt_engine(onnx_path, fp16=True):
        """Build a TensorRT engine from ONNX. Handles TRT 8/9/10 APIs."""
        builder = trt.Builder(TRT_LOGGER)

        # TRT 10 removed the EXPLICIT_BATCH flag (all networks are explicit-batch by default)
        if TRT_MAJOR >= 10:
            network = builder.create_network()
        else:
            network = builder.create_network(
                1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
            )

        parser = trt.OnnxParser(network, TRT_LOGGER)
        with open(onnx_path, 'rb') as f:
            if not parser.parse(f.read()):
                for i in range(parser.num_errors):
                    print(f'  Parse error [{i}]: {parser.get_error(i)}')
                return None

        config = builder.create_builder_config()
        config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 2 << 30)  # 2 GB

        if fp16 and builder.platform_has_fast_fp16:
            config.set_flag(trt.BuilderFlag.FP16)
            print('✓ FP16 mode enabled')

        # Optimization profile: required when the ONNX has dynamic dims (-1).
        # We exported with static shapes, but add a profile defensively.
        if network.num_inputs > 0:
            inp = network.get_input(0)
            shape = tuple(d if d > 0 else 1 for d in inp.shape)
            profile = builder.create_optimization_profile()
            profile.set_shape(inp.name, shape, shape, shape)
            config.add_optimization_profile(profile)

        print('Building TensorRT engine (this may take a few minutes)...')
        serialized = builder.build_serialized_network(network, config)
        if serialized is None:
            print('⚠ FP16 build failed, retrying FP32...')
            config.clear_flag(trt.BuilderFlag.FP16)
            serialized = builder.build_serialized_network(network, config)
        if serialized is None:
            print('✗ Engine build failed.')
            return None

        runtime = trt.Runtime(TRT_LOGGER)
        engine  = runtime.deserialize_cuda_engine(serialized)
        print('✓ Engine built successfully')
        return engine

    engine = build_trt_engine(ONNX_TRT_PATH, fp16=True)  # opset-17 file
else:
    engine = None


In [ ]:
if trt_available and engine is not None:
    class TRTInference:
        """Version-aware TensorRT inference wrapper (TRT 8 / 9 / 10)."""

        def __init__(self, engine):
            self.context = engine.create_execution_context()
            self.trt_major = int(trt.__version__.split('.')[0])

            # Shape API differs between TRT versions
            if self.trt_major >= 10:
                in_shape  = tuple(engine.get_tensor_shape('input'))
                out_shape = tuple(engine.get_tensor_shape('output'))
            else:
                in_shape  = tuple(engine.get_binding_shape(0))
                out_shape = tuple(engine.get_binding_shape(1))

            self.d_input      = cuda.mem_alloc(int(np.prod(in_shape))  * 4)
            self.d_output     = cuda.mem_alloc(int(np.prod(out_shape)) * 4)
            self.output_shape = out_shape
            self.stream       = cuda.Stream()

        def infer(self, input_data):
            inp = np.ascontiguousarray(input_data, dtype=np.float32)
            out = np.empty(self.output_shape, dtype=np.float32)

            cuda.memcpy_htod_async(self.d_input, inp, self.stream)

            if self.trt_major >= 10:
                self.context.set_tensor_address('input',  int(self.d_input))
                self.context.set_tensor_address('output', int(self.d_output))
                self.context.execute_async_v3(stream_handle=self.stream.handle)
            else:
                bindings = [int(self.d_input), int(self.d_output)]
                self.context.execute_async_v2(bindings=bindings,
                                              stream_handle=self.stream.handle)

            cuda.memcpy_dtoh_async(out, self.d_output, self.stream)
            self.stream.synchronize()
            return out

    trt_infer = TRTInference(engine)

    # Verify output
    trt_output = trt_infer.infer(onnx_input)
    trt_seg, _ = visualize_segmentation(trt_output)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    axes[0].imshow(pytorch_seg); axes[0].set_title('PyTorch'); axes[0].axis('off')
    axes[1].imshow(trt_seg);     axes[1].set_title('TensorRT FP16'); axes[1].axis('off')
    plt.suptitle('PyTorch vs TensorRT Output', fontsize=14)
    plt.tight_layout(); plt.show()

    trt_time = benchmark(lambda: trt_infer.infer(onnx_input), 'TensorRT (GPU, FP16)')
else:
    trt_time = None
    print('Skipping TensorRT (not available on this runtime)')


---
## 8. Production Profiling & Benchmarking

### Key Metrics
- **Latency:** Time to process one inference (ms)
- **Throughput:** Inferences per second (FPS)
- **Memory:** GPU/CPU memory used during inference
- **Model size:** Disk space (affects deployment size)

In [ ]:
# Summary of all benchmarks
print('\n' + '='*70)
print('  DEPLOYMENT BENCHMARK RESULTS')
print('  Model: DeepLabV3-MobileNetV3-Large')
print('='*70)
print(f'  {"Framework":<38} {"ms":<10} {"FPS":<10} {"Speedup":<8}')
print(f'  {"-"*38} {"-"*10} {"-"*10} {"-"*8}')

rows = [
    ('PyTorch CPU (baseline)',   pytorch_time),
    ('ONNX Runtime CPU',         onnx_time),
    ('ONNX Runtime CPU (pruned)', pruned_onnx_time),
    ('ONNX Runtime CPU (INT8)',  quant_onnx_time),
]
if trt_time:
    rows.append(('TensorRT GPU FP16', trt_time))

for name, t in rows:
    print(f'  {name:<38} {t:<10.1f} {1000/t:<10.1f} {pytorch_time/t:<8.2f}x')

print()
print(f'  Model sizes:')
print(f'    FP32 ONNX:  {original_size:.1f} MB')
print(f'    INT8 ONNX:  {quant_size:.1f} MB  ({(1-quant_size/original_size)*100:.0f}% smaller)')
print('='*70)


---
## 9. Full Pipeline Project: Train → Optimize → Export → Deploy → Benchmark

### End-to-End Autonomous Vehicle Perception

This section demonstrates the **complete production workflow**:

```
1. TRAIN      → Define and train a PyTorch model
2. OPTIMIZE   → Prune and quantize
3. EXPORT     → Convert to ONNX
4. DEPLOY     → Load with ONNX Runtime / TensorRT
5. BENCHMARK  → Compare all frameworks
```

In [ ]:
# Project: Simple object detector (YOLO-style)
print('PROJECT: Simple Object Detector')
print('Goal: Train → Optimize → Export → Deploy → Benchmark')
print('\nStep 1: TRAIN')
print('-' * 60)

class SimpleDetector(torch_nn.Module):
    """Tiny object detection model."""
    def __init__(self):
        super().__init__()
        self.backbone = torch_nn.Sequential(
            torch_nn.Conv2d(3, 16, 3, padding=1),
            torch_nn.ReLU(),
            torch_nn.MaxPool2d(2),   # 416→208
            torch_nn.Conv2d(16, 32, 3, padding=1),
            torch_nn.ReLU(),
            torch_nn.MaxPool2d(2),   # 208→104
        )
        self.head = torch_nn.Conv2d(32, 5, 1)  # [tx, ty, tw, th, conf] at 104x104

    def forward(self, x):
        return self.head(self.backbone(x))

detector = SimpleDetector()
detector.train()
optimizer = torch.optim.Adam(detector.parameters())

# Compute actual output shape before creating targets
with torch.no_grad():
    _sample = detector(torch.zeros(1, 3, 416, 416))
    _out_shape = _sample.shape[1:]  # (5, H, W)

print(f'Model: SimpleDetector ({sum(p.numel() for p in detector.parameters()):,} params)')
print(f'Input: (batch, 3, 416, 416)')
print(f'Output: (batch, {_out_shape[0]}, {_out_shape[1]}, {_out_shape[2]})  # predictions per cell')
print()

detector.train()
for epoch in range(3):
    synthetic_batch = torch.randn(4, 3, 416, 416)
    targets = torch.randn(4, *_out_shape)   # match real output shape

    outputs = detector(synthetic_batch)
    loss = ((outputs - targets) ** 2).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f'  Epoch {epoch+1}/3  Loss: {loss.item():.4f}')

detector.eval()
print('✓ Training complete')


In [ ]:
print('\nStep 2: OPTIMIZE')
print('-' * 60)

# Pruning (unstructured L1 - keeps tensor shapes intact)
optimized_detector = copy.deepcopy(detector)

det_conv_params = [(m, 'weight') for m in optimized_detector.modules() if isinstance(m, torch_nn.Conv2d)]
prune.global_unstructured(det_conv_params, pruning_method=prune.L1Unstructured, amount=0.3)
for m, name in det_conv_params:
    prune.remove(m, name)

total_w = sum(p.numel() for p in detector.parameters())
nonzero_w = sum((p != 0).sum().item() for p in optimized_detector.parameters())
print(f'Sparsity: {100*(1 - nonzero_w/total_w):.1f}% weights zeroed')
print('✓ Optimization complete')


In [ ]:
print('\nStep 3: EXPORT')
print('-' * 60)

DETECTOR_ONNX = 'simple_detector.onnx'

dummy_det_input = torch.randn(1, 3, 416, 416)
torch.onnx.export(
    optimized_detector,
    dummy_det_input,
    DETECTOR_ONNX,
    input_names=['image'],
    output_names=['detections'],
    opset_version=18
)

model_det = onnx.load(DETECTOR_ONNX)
onnx.checker.check_model(model_det)

det_size = os.path.getsize(DETECTOR_ONNX) / 1024
print(f'Exported to: {DETECTOR_ONNX}')
print(f'File size: {det_size:.1f} KB')
print('✓ Export complete')


In [ ]:
print('\nStep 4: DEPLOY')
print('-' * 60)

det_session = ort.InferenceSession(DETECTOR_ONNX, providers=['CPUExecutionProvider'])
det_input = det_session.get_inputs()[0]
det_output = det_session.get_outputs()[0]

test_det_image = np.random.randn(1, 3, 416, 416).astype(np.float32)
det_pred = det_session.run([det_output.name], {det_input.name: test_det_image})[0]

print(f'Input shape:  {test_det_image.shape}')
print(f'Output shape: {det_pred.shape}')
print('✓ Deployment complete (ONNX Runtime)')


In [ ]:
print('\nStep 5: BENCHMARK')
print('-' * 60)

# PyTorch baseline
pytorch_det_time = benchmark(
    lambda: detector(dummy_det_input),
    'PyTorch (CPU)'
)

# ONNX Runtime
onnx_det_time = benchmark(
    lambda: det_session.run([det_output.name], {det_input.name: test_det_image}),
    'ONNX Runtime (CPU)'
)

print(f'\nSpeedup: {pytorch_det_time / onnx_det_time:.2f}x faster with ONNX')
print('\n' + '='*70)
print('  FULL PIPELINE COMPLETE')
print('='*70)
print('\nWhat you learned:')
print('  1. How to train and optimize models')
print('  2. Export PyTorch to production-ready ONNX')
print('  3. Deploy with multiple runtimes')
print('  4. Benchmark and compare performance')
print('  5. This is the workflow Autoware uses')
print('='*70)


---

## Conclusion & Next Steps

### You've learned:
✓ **Load & benchmark** a PyTorch robotics model  
✓ **Export to ONNX** with full validation  
✓ **Deploy with ONNX Runtime** and TensorRT  
✓ **Optimize** via pruning & quantization before export  
✓ **Analyze Autoware's production C++ code**  
✓ **Complete end-to-end pipeline**

### In Production (Autoware, Tesla, Waymo):
- Train in PyTorch with optimizations from the **Neural Optimization** course
- Export to ONNX for deployment flexibility
- Use TensorRT on Nvidia GPUs (fastest)
- Use ONNX Runtime on CPUs (most portable)
- Package into ROS2 nodes (Autoware pattern)

### Further Learning:
- **Neural Optimization course** — Master pruning, quantization, distillation
- **Autoware documentation** — Deploy models in real autonomous driving stack
- **TensorRT documentation** — Advanced optimization techniques
- **ONNX Zoo** — Pre-trained models for various tasks

---

*The Deployment Notebook — Neural Optimization by Think Autonomous*

https://thinkautonomous.ai
